# OpenSecrets Dark Money Target Pipeline

This notebook walks through our process of manually generating verified dark money organization targets within our database based on OpenSecret's [Top Election Spenders](https://www.opensecrets.org/dark-money/top-election-spenders) data. This pipeline required the following process:
1. Copying all of the top spenders for each cycle going back to 1990 into a Google Sheet, and downloading into our repo
2. Reading in that list of organizations, and using ProPublica's API to systematically search for potential matches
3. Exporting those potential matches to a csv. We uploaded them back into Google Sheets, but any tabular editing tool would suffice.
4. Manually verify each potential match from the table of data generated in step 3. The rationale, and supporting evidence, was documented
5. After generating those targets, filter down to a unique list of verified EIN-Name pairs and export into S3

In [1]:
import re
import sqlite3
import sys
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    
from extract.config import settings
from extract.s3_manager import df_to_s3

In [2]:
conn = sqlite3.connect(settings.db_path)

## Import Top Spenders Data

Note that the table was generated externally, and this is the second part of the first step as outlined above.

In [3]:
orgs = pd.read_csv("../data/open_secret_top_spenders.csv").loc[:, ['Organization', 'Type']].drop_duplicates()
print(f"There are {len(orgs.Organization.unique())} unique organization names.")
orgs.head()

There are 463 unique organization names.


,Organization,Type
0,Americans for Constitutional Liberty,501(c)(4)
1,Ohio Works,501(c)(4)
2,Campaign for a Family Friendly Economy,501(c)(4)
3,National Assn of Realtors,501(c)(6)
4,Defending Democracy Together,501(c)(4)


## Generate & Export Potential Matches

In [9]:
type_pattern = re.compile(r'501\(c\)\((\d{1,2})\)')
match_df_cols = ['org', 'org_type', 'ein', 'name', 'exempt_type', 'match_confidence', 'filer_ein', 'filer_name', 'filer_name_norm', 'method', 'verified', 'details']
match_df_rows = []
top_n_matches = 3
for row in orgs.values:
    org = row[0]
    org_type = row[1]
    org_type = type_pattern.search(org_type)
    if org_type:
        org_type = int(org_type.group(1))

    url = f"https://projects.propublica.org/nonprofits/api/v2/search.json?q={org}"
    r = requests.get(url).json()

    if r['total_results'] == 0:
        match_df_row = [
            org, org_type,
            None, None, None, None, None, None, None, # 'ein', 'name', 'exempt_type', 'match_confidence', 'filer_ein', 'filer_name', 'filer_name_norm',
            "auto", 0, "Not found in ProPublica's NonProfit Explorer search method." # 'method', 'verified', 'details'
        ]
        match_df_rows.append(match_df_row)
        continue

    matches = r['organizations']
    matches = [
        [d['ein'], d['name'], d['subseccd'], d['score']]
        for d in matches
    ]

    for i, match in enumerate(matches):
        org_record = pd.read_sql(f"select ein, current_name, normalized_name from organizations where ein = {match[0]}", conn)
        if org_record.empty:
            org_record = [None, None, None] # 'filer_ein', 'filer_name', 'filer_name_norm'
        else:
            org_record = org_record.values.tolist()[0]

        # The first several elements won't change after this point
        match_df_row = [
            org, org_type,
            *match, # 'ein', 'name', 'exempt_type', 'match_confidence'
            *org_record, # 'filer_name', 'filer_name_norm'
        ]

        if i >= top_n_matches:
            match_df_row += [
                "auto", 0, f"Not included in top {top_n_matches} results." # 'method', 'verified', 'details'
            ]
            match_df_rows.append(match_df_row)
            continue

        match_df_row += [
            "manual", None, "" # 'method', 'verified', 'details'
        ]
        match_df_rows.append(match_df_row)
        
match_df = pd.DataFrame(match_df_rows, columns=match_df_cols)
match_df

,org,org_type,ein,name,exempt_type,match_confidence,filer_ein,filer_name,filer_name_norm,method,verified,details
0,Americans for Constitutional Liberty,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,auto,0.0,Not found in ProPublica's NonProfit Explorer s...
1,Ohio Works,4.0,991805086.0,Ohio Works,NaN,97.884690,991805086,Ohio Works,ohio works,manual,NaN,
2,Ohio Works,4.0,820990131.0,Ohio Works,4.0,97.884690,NaN,NaN,NaN,manual,NaN,
3,Ohio Works,4.0,341900652.0,Alliance Of Ohio Work Centers,4.0,62.358486,NaN,NaN,NaN,manual,NaN,
4,Ohio Works,4.0,331202355.0,Ohio School Social Work Association,6.0,62.358486,NaN,NaN,NaN,auto,0.0,Not included in top 3 results.
...,...,...,...,...,...,...,...,...,...,...,...,...
1905,Michigan Right to Life,4.0,351594988.0,Right To Life Michiana Education Fund Inc,3.0,105.467995,351594988,RIGHT TO LIFE MICHIANA EDUCATION,right to life michiana education,auto,0.0,Not included in top 3 results.
1906,Michigan Right to Life,4.0,383144453.0,Right To Life Of Michigan Educational Endowmen...,3.0,104.884580,383144453,RIGHT TO LIFE OF MICHIGAN,right to life michigan,auto,0.0,Not included in top 3 results.
1907,Michigan Right to Life,4.0,382795894.0,Right To Life Of Michigan Educational Foundati...,3.0,104.884580,NaN,NaN,NaN,auto,0.0,Not included in top 3 results.
1908,California Assn Of Realtors,6.0,363535493.0,Institute Of Real Estate Management Of The Nat...,6.0,23.974482,NaN,NaN,NaN,manual,NaN,


In [8]:
match_df.to_csv('../data/potential_dark_money_matches.csv', index=False)

## Ingest the Verification Sheet

In [4]:
# Enforce column typing
num_cols = ['org_type', 'exempt_type', 'verified']
matching_matrix = pd.read_csv('../data/matching_matrix.csv')
matching_matrix_schema = {col: float if col in num_cols else str for col in matching_matrix.columns}
matching_matrix = matching_matrix.astype(matching_matrix_schema)
print(f"There are now {len(matching_matrix)} records.")
matching_matrix.head()

There are now 1937 records.


,org,org_type,ein,name,exempt_type,match_confidence,filer_ein,filer_name,filer_name_norm,method,verified,details,date_accessed
0,215 People's Alliance,4.0,813511044.0,215 Peoples Alliance,4.0,144.59,813511044.0,215 Peoples Alliance,215 peoples alliance,auto - reviewed,1.0,"Exact matches across the board, and only match...",2026-08-06
1,350.org Action Fund,4.0,261181604.0,350 Org Action Fund,4.0,173.65,261181604.0,350ORG ACTION FUND,350org action fund,auto - reviewed,1.0,"Exact matches across the board, and only match...",2026-08-06
2,45 Cmte,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,manual,1.0,Although not found in ProPublica's NonProfit E...,2026-08-06
3,60 Plus Assn,4.0,911660549.0,Vancouver Metro Senior 60 Plus Softball Associ...,3.0,26.95,NaN,NaN,NaN,manual,0.0,The OpenSecret link below reveals a different ...,2026-08-06
4,60 Plus Assn,NaN,NaN,NaN,NaN,NaN,541564919.0,THE 60 PLUS ASSOCIATION INC,60 plus association,manual,1.0,The OpenSecret link below revealed this EIN: h...,2026-08-06


## Filter Out Pre-2012 Labels

To account for weak model performance, we are removing training labels pertaining to 2010 cycle and before. 2010 was selected as that was when the Supreme Court decisions were made that led to greater dark money activity.

In [7]:
CUTOFF = 2010

In [16]:
post_2010_orgs = pd.read_csv("../data/open_secret_top_spenders.csv").loc[:, ['Organization', 'Type', 'Cycle']]
post_2010_orgs = post_2010_orgs[post_2010_orgs.Cycle.gt(CUTOFF)]
post_2010_orgs

,Organization,Type,Cycle
0,Americans for Constitutional Liberty,501(c)(4),2024
1,Ohio Works,501(c)(4),2024
2,Campaign for a Family Friendly Economy,501(c)(4),2024
3,National Assn of Realtors,501(c)(6),2024
4,Defending Democracy Together,501(c)(4),2024
...,...,...,...
640,Enterprise Freedom Action Cmte,501(c)(4),2012
641,Iowa Credit Union League,501(c)(6),2012
642,Planned Parenthood/Pacific Southwest,501(c)(4),2012
643,Women's Voices Women Vote Action Fund,501(c)(4),2012


In [17]:
post_2010_orgs = post_2010_orgs.loc[:, ['Organization']].drop_duplicates()
print(f"There are {len(post_2010_orgs.Organization.unique())} unique organization names pertaining to post-{CUTOFF} organizations.")
post_2010_orgs = post_2010_orgs.Organization.values
post_2010_orgs

There are 394 unique organization names pertaining to post-2010 organizations.


<ArrowStringArray>
[    'Americans for Constitutional Liberty',
                               'Ohio Works',
   'Campaign for a Family Friendly Economy',
                'National Assn of Realtors',
             'Defending Democracy Together',
       'Black Progressive Action Coalition',
                'Building America's Future',
           'Progress North Carolina Action',
                     'Montana Rural Voters',
                           'Somos Votantes',
 ...
 'California League of Conservation Voters',
                      'Common Sense Issues',
                   'Colorado Family Action',
                  'Wisconsin Family Action',
  'Nevada Advocates for Planned Parenthood',
                     'A Better America Now',
           'Americans United 4 Life Action',
                    'America is not Stupid',
                 'Iowa Credit Union League',
                      'Direct Selling Assn']
Length: 394, dtype: str

## Generate & Export Unique List of Verified Targets

For the purposes of incorporating these labels into the broader pipeline, only the `filer_ein` is necessary. For the purposes of context, the `filer_name` is included in this export.

In [18]:
verified_matches = (matching_matrix.verified == 1) & (matching_matrix.filer_ein.notna()) & (matching_matrix.org.isin(post_2010_orgs))
open_secrets_dark_money_orgs = matching_matrix.loc[verified_matches, ['filer_ein', 'filer_name']].drop_duplicates()
print(f"There are {len(open_secrets_dark_money_orgs)} unique EIN-name pairings.")
open_secrets_dark_money_orgs.head()

There are 317 unique EIN-name pairings.


,filer_ein,filer_name
0,813511044.0,215 Peoples Alliance
1,261181604.0,350ORG ACTION FUND
4,541564919.0,THE 60 PLUS ASSOCIATION INC
6,844281129.0,A STRONGER LOUISIANA
32,272301175.0,Action For A Progressive Future


In [19]:
df_to_s3(open_secrets_dark_money_orgs, 'parquet_updated/dark_money_labels.parquet')